# Ungraded Lab: Walkthrough of ML Metadata

Keeping records at each stage of the project is an important aspect of machine learning pipelines. [ML Metadata](https://www.tensorflow.org/tfx/guide/mlmd) addresses this need by having an API suited specifically for keeping track of any progress made in ML projects.

In this notebook, we implement the **same data model and concepts** as MLMD — ArtifactTypes, Artifacts, ExecutionTypes, Executions, Events, Contexts, Attributions, and Associations — using a **lightweight SQLite-based metadata store** built from scratch. This approach demonstrates the core ideas without requiring TensorFlow or TFDV dependencies.

The pipeline covers 4 stages, all fully tracked:
1. **Data Validation** — Infer a schema from training data
2. **Anomaly Detection** — Validate eval data against the schema
3. **Model Training** — Train an SVM classifier on the Wine dataset
4. **Model Evaluation** — Compute accuracy, F1, precision, and recall

Let's get to it!

## Imports

In [ ]:
import sqlite3
import json
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datetime import datetime

print('All imports successful')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

## Load and Split the Wine Dataset

The [Wine dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) contains 178 samples with 13 chemical features, classified into 3 wine cultivars. We split it into train (60%), eval (20%), and serving (20%) sets.

In [ ]:
# Load the Wine dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')
data = pd.concat([X, y], axis=1)

# Split: 60% train, 20% eval, 20% serving
train, remainder = train_test_split(data, train_size=0.6, random_state=42)
eval_data, serving = train_test_split(remainder, test_size=0.5, random_state=42)

# Save splits to CSV
for split_name, split_df in [('train', train), ('eval', eval_data), ('serving', serving)]:
    split_path = os.path.join('data', split_name)
    os.makedirs(split_path, exist_ok=True)
    split_df.to_csv(os.path.join(split_path, 'data.csv'), index=False)

print(f'Wine dataset split: {len(train)} train, {len(eval_data)} eval, {len(serving)} serving')
print(f'Features: {list(wine.feature_names)}')
print(f'Classes: {list(wine.target_names)}')

## Process Outline

We implement the same data model as ML Metadata:

* **ArtifactType** — describes an artifact's type and properties
* **Artifact** — a specific instance (dataset, schema, model, etc.)
* **ExecutionType** — describes a pipeline step type
* **Execution** — a record of a pipeline step run
* **Event** — relationship between artifacts and executions (INPUT/OUTPUT)
* **ContextType** — describes a grouping type (e.g., experiment)
* **Context** — groups artifacts and executions together
* **Attribution** — links artifacts to contexts
* **Association** — links executions to contexts

All stored in a persistent **SQLite database** at `metadata/mlmd.sqlite`.

## Define ML Metadata's Storage Database

We create a SQLite database with tables mirroring the MLMD data model. This persists across sessions, just like a real MLMD store.

In [ ]:
# Setup persistent SQLite metadata store
os.makedirs('./metadata', exist_ok=True)

# Remove existing DB to start fresh
db_path = './metadata/mlmd.sqlite'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create tables mirroring the MLMD data model
cursor.executescript('''
    CREATE TABLE artifact_types (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        properties TEXT
    );
    CREATE TABLE artifacts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type_id INTEGER NOT NULL,
        uri TEXT,
        properties TEXT,
        create_time TEXT DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (type_id) REFERENCES artifact_types(id)
    );
    CREATE TABLE execution_types (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        properties TEXT
    );
    CREATE TABLE executions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type_id INTEGER NOT NULL,
        state TEXT,
        create_time TEXT DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (type_id) REFERENCES execution_types(id)
    );
    CREATE TABLE events (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        artifact_id INTEGER NOT NULL,
        execution_id INTEGER NOT NULL,
        type TEXT NOT NULL,
        create_time TEXT DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (artifact_id) REFERENCES artifacts(id),
        FOREIGN KEY (execution_id) REFERENCES executions(id)
    );
    CREATE TABLE context_types (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        properties TEXT
    );
    CREATE TABLE contexts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        type_id INTEGER NOT NULL,
        name TEXT UNIQUE NOT NULL,
        properties TEXT,
        create_time TEXT DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (type_id) REFERENCES context_types(id)
    );
    CREATE TABLE attributions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        artifact_id INTEGER NOT NULL,
        context_id INTEGER NOT NULL,
        FOREIGN KEY (artifact_id) REFERENCES artifacts(id),
        FOREIGN KEY (context_id) REFERENCES contexts(id)
    );
    CREATE TABLE associations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        execution_id INTEGER NOT NULL,
        context_id INTEGER NOT NULL,
        FOREIGN KEY (execution_id) REFERENCES executions(id),
        FOREIGN KEY (context_id) REFERENCES contexts(id)
    );
''')
conn.commit()

print(f'Metadata store created at: {db_path}')
print('Tables:', [row[0] for row in cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

## Helper Functions

These functions mirror the MLMD API: `put_artifact_type()`, `put_artifacts()`, `put_execution_type()`, `put_executions()`, `put_events()`, etc.

In [ ]:
def put_artifact_type(name, properties):
    cursor.execute('INSERT INTO artifact_types (name, properties) VALUES (?, ?)',
                   (name, json.dumps(properties)))
    conn.commit()
    return cursor.lastrowid

def put_artifact(type_id, uri, properties):
    cursor.execute('INSERT INTO artifacts (type_id, uri, properties) VALUES (?, ?, ?)',
                   (type_id, uri, json.dumps(properties)))
    conn.commit()
    return cursor.lastrowid

def put_execution_type(name, properties):
    cursor.execute('INSERT INTO execution_types (name, properties) VALUES (?, ?)',
                   (name, json.dumps(properties)))
    conn.commit()
    return cursor.lastrowid

def put_execution(type_id, state='RUNNING'):
    cursor.execute('INSERT INTO executions (type_id, state) VALUES (?, ?)',
                   (type_id, state))
    conn.commit()
    return cursor.lastrowid

def update_execution(exec_id, state):
    cursor.execute('UPDATE executions SET state=? WHERE id=?', (state, exec_id))
    conn.commit()

def put_event(artifact_id, execution_id, event_type):
    cursor.execute('INSERT INTO events (artifact_id, execution_id, type) VALUES (?, ?, ?)',
                   (artifact_id, execution_id, event_type))
    conn.commit()
    return cursor.lastrowid

def put_context_type(name, properties):
    cursor.execute('INSERT INTO context_types (name, properties) VALUES (?, ?)',
                   (name, json.dumps(properties)))
    conn.commit()
    return cursor.lastrowid

def put_context(type_id, name, properties):
    cursor.execute('INSERT INTO contexts (type_id, name, properties) VALUES (?, ?, ?)',
                   (type_id, name, json.dumps(properties)))
    conn.commit()
    return cursor.lastrowid

def put_attribution(artifact_id, context_id):
    cursor.execute('INSERT INTO attributions (artifact_id, context_id) VALUES (?, ?)',
                   (artifact_id, context_id))
    conn.commit()

def put_association(execution_id, context_id):
    cursor.execute('INSERT INTO associations (execution_id, context_id) VALUES (?, ?)',
                   (execution_id, context_id))
    conn.commit()

# Query helpers
def get_artifact(artifact_id):
    row = cursor.execute('SELECT * FROM artifacts WHERE id=?', (artifact_id,)).fetchone()
    return {'id': row[0], 'type_id': row[1], 'uri': row[2], 'properties': json.loads(row[3]), 'create_time': row[4]}

def get_artifacts_by_type(type_name):
    rows = cursor.execute('''
        SELECT a.* FROM artifacts a
        JOIN artifact_types t ON a.type_id = t.id
        WHERE t.name = ?
    ''', (type_name,)).fetchall()
    return [{'id': r[0], 'type_id': r[1], 'uri': r[2], 'properties': json.loads(r[3]), 'create_time': r[4]} for r in rows]

def get_events_by_artifact(artifact_id):
    rows = cursor.execute('SELECT * FROM events WHERE artifact_id=?', (artifact_id,)).fetchall()
    return [{'id': r[0], 'artifact_id': r[1], 'execution_id': r[2], 'type': r[3], 'create_time': r[4]} for r in rows]

def get_events_by_execution(execution_id):
    rows = cursor.execute('SELECT * FROM events WHERE execution_id=?', (execution_id,)).fetchall()
    return [{'id': r[0], 'artifact_id': r[1], 'execution_id': r[2], 'type': r[3], 'create_time': r[4]} for r in rows]

def get_artifact_type_name(type_id):
    return cursor.execute('SELECT name FROM artifact_types WHERE id=?', (type_id,)).fetchone()[0]

def get_execution_type_name(type_id):
    return cursor.execute('SELECT name FROM execution_types WHERE id=?', (type_id,)).fetchone()[0]

def get_artifacts_by_context(context_id):
    rows = cursor.execute('''
        SELECT a.* FROM artifacts a
        JOIN attributions attr ON a.id = attr.artifact_id
        WHERE attr.context_id = ?
    ''', (context_id,)).fetchall()
    return [{'id': r[0], 'type_id': r[1], 'uri': r[2], 'properties': json.loads(r[3]), 'create_time': r[4]} for r in rows]

def get_executions_by_context(context_id):
    rows = cursor.execute('''
        SELECT e.* FROM executions e
        JOIN associations assoc ON e.id = assoc.execution_id
        WHERE assoc.context_id = ?
    ''', (context_id,)).fetchall()
    return [{'id': r[0], 'type_id': r[1], 'state': r[2], 'create_time': r[3]} for r in rows]

print('Helper functions defined')

## Register ArtifactTypes

We register 6 artifact types: DataSet, Schema, Statistics, Anomalies, Model, and ModelEvaluation.

In [ ]:
# Register artifact types
statistics_type_id = put_artifact_type('Statistics', {'name': 'STRING', 'split': 'STRING', 'version': 'STRING'})
data_type_id = put_artifact_type('DataSet', {'name': 'STRING', 'split': 'STRING', 'version': 'INT'})
schema_type_id = put_artifact_type('Schema', {'name': 'STRING', 'version': 'INT'})
anomaly_type_id = put_artifact_type('Anomalies', {'name': 'STRING', 'num_anomalies': 'INT', 'description': 'STRING'})
model_type_id = put_artifact_type('Model', {'name': 'STRING', 'version': 'INT', 'framework': 'STRING'})
eval_type_id = put_artifact_type('ModelEvaluation', {'name': 'STRING', 'accuracy': 'DOUBLE', 'f1_score': 'DOUBLE', 'precision': 'DOUBLE', 'recall': 'DOUBLE'})

print(f'DataSet type ID: {data_type_id}')
print(f'Schema type ID: {schema_type_id}')
print(f'Statistics type ID: {statistics_type_id}')
print(f'Anomalies type ID: {anomaly_type_id}')
print(f'Model type ID: {model_type_id}')
print(f'ModelEvaluation type ID: {eval_type_id}')

## Register ExecutionTypes

We register 4 execution types for the pipeline stages.

In [ ]:
dv_exec_type_id = put_execution_type('Data Validation', {'state': 'STRING'})
anomaly_exec_type_id = put_execution_type('Anomaly Detection', {'state': 'STRING'})
training_exec_type_id = put_execution_type('Model Training', {'state': 'STRING'})
eval_exec_type_id = put_execution_type('Model Evaluation', {'state': 'STRING'})

print(f'Data Validation exec type ID: {dv_exec_type_id}')
print(f'Anomaly Detection exec type ID: {anomaly_exec_type_id}')
print(f'Model Training exec type ID: {training_exec_type_id}')
print(f'Model Evaluation exec type ID: {eval_exec_type_id}')

## Stage 1: Data Validation

We infer a schema from the training data by analyzing column types, min/max values, and unique counts. This mirrors what TFDV does internally.

In [ ]:
# Register input artifact
train_artifact_id = put_artifact(data_type_id, './data/train/data.csv',
                                  {'name': 'Wine dataset', 'split': 'train', 'version': 1})

# Create execution and register input event
dv_exec_id = put_execution(dv_exec_type_id, 'RUNNING')
put_event(train_artifact_id, dv_exec_id, 'DECLARED_INPUT')

print(f'Train artifact ID: {train_artifact_id}')
print(f'Data Validation execution ID: {dv_exec_id}')

In [ ]:
# Infer schema from training data
train_df = pd.read_csv('./data/train/data.csv')

schema = {}
for col in train_df.columns:
    col_info = {
        'dtype': str(train_df[col].dtype),
        'min': float(train_df[col].min()),
        'max': float(train_df[col].max()),
        'mean': float(train_df[col].mean()),
        'null_count': int(train_df[col].isnull().sum()),
        'unique_count': int(train_df[col].nunique())
    }
    schema[col] = col_info

# Save schema
schema_file = './schema.json'
with open(schema_file, 'w') as f:
    json.dump(schema, f, indent=2)

print(f'Schema inferred with {len(schema)} features')
print(f'Schema saved to: {schema_file}')
for col, info in list(schema.items())[:3]:
    print(f'  {col}: dtype={info["dtype"]}, range=[{info["min"]:.2f}, {info["max"]:.2f}]')
print('  ...')

In [ ]:
# Register schema artifact and output event
schema_artifact_id = put_artifact(schema_type_id, schema_file,
                                   {'name': 'Wine Schema', 'version': 1})
put_event(schema_artifact_id, dv_exec_id, 'DECLARED_OUTPUT')

# Mark execution as completed
update_execution(dv_exec_id, 'COMPLETED')

print(f'Schema artifact ID: {schema_artifact_id}')
print('Data Validation execution: COMPLETED')

## Stage 2: Anomaly Detection

We validate the eval dataset against the schema. We check for: values outside the training range, unexpected null values, and new categories.

In [ ]:
# Register eval artifact and anomaly detection execution
eval_anomaly_artifact_id = put_artifact(data_type_id, './data/eval/data.csv',
                                         {'name': 'Wine dataset', 'split': 'eval', 'version': 1})

anomaly_exec_id = put_execution(anomaly_exec_type_id, 'RUNNING')

# Input events: eval dataset + schema
put_event(eval_anomaly_artifact_id, anomaly_exec_id, 'DECLARED_INPUT')
put_event(schema_artifact_id, anomaly_exec_id, 'DECLARED_INPUT')

print(f'Eval artifact ID (for anomaly detection): {eval_anomaly_artifact_id}')
print(f'Anomaly Detection execution ID: {anomaly_exec_id}')

In [ ]:
# Run anomaly detection: validate eval data against schema
eval_df = pd.read_csv('./data/eval/data.csv')

anomalies_found = []
for col, col_schema in schema.items():
    if col not in eval_df.columns:
        anomalies_found.append(f'{col}: MISSING FEATURE')
        continue
    
    eval_min = float(eval_df[col].min())
    eval_max = float(eval_df[col].max())
    eval_nulls = int(eval_df[col].isnull().sum())
    
    if eval_min < col_schema['min']:
        anomalies_found.append(f'{col}: value {eval_min:.2f} below training min {col_schema["min"]:.2f}')
    if eval_max > col_schema['max']:
        anomalies_found.append(f'{col}: value {eval_max:.2f} above training max {col_schema["max"]:.2f}')
    if eval_nulls > 0 and col_schema['null_count'] == 0:
        anomalies_found.append(f'{col}: {eval_nulls} unexpected nulls')

# Check for extra columns in eval
for col in eval_df.columns:
    if col not in schema:
        anomalies_found.append(f'{col}: UNEXPECTED FEATURE')

# Save anomalies report
anomalies_path = './anomalies.json'
anomaly_report = {'num_anomalies': len(anomalies_found), 'anomalies': anomalies_found}
with open(anomalies_path, 'w') as f:
    json.dump(anomaly_report, f, indent=2)

if anomalies_found:
    print(f'Found {len(anomalies_found)} anomalies in eval data:')
    for a in anomalies_found:
        print(f'  - {a}')
else:
    print('No anomalies found — eval data conforms to the schema.')

print(f'\nAnomalies report saved to: {anomalies_path}')

In [ ]:
# Record anomaly artifact and complete execution
anomaly_desc = '; '.join(anomalies_found) if anomalies_found else 'No anomalies detected'
anomaly_artifact_id = put_artifact(anomaly_type_id, anomalies_path,
                                    {'name': 'Wine Eval Anomalies',
                                     'num_anomalies': len(anomalies_found),
                                     'description': anomaly_desc})

put_event(anomaly_artifact_id, anomaly_exec_id, 'DECLARED_OUTPUT')
update_execution(anomaly_exec_id, 'COMPLETED')

print(f'Anomaly artifact ID: {anomaly_artifact_id}')
print('Anomaly Detection execution: COMPLETED')

## Stage 3: Model Training

We train an SVM classifier on the Wine training data and track it through our metadata store.

In [ ]:
# Create training execution and register input
training_exec_id = put_execution(training_exec_type_id, 'RUNNING')
put_event(train_artifact_id, training_exec_id, 'DECLARED_INPUT')

print(f'Model Training execution ID: {training_exec_id}')

In [ ]:
# Train SVM classifier
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

# Scale features (critical for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train model
model = SVC(kernel='rbf', C=1.0, random_state=42)
model.fit(X_train_scaled, y_train)

# Save model and scaler
os.makedirs('./model', exist_ok=True)
model_path = './model/model.pkl'
scaler_path = './model/scaler.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f'Model trained on {len(X_train)} samples with {X_train.shape[1]} features')
print(f'Training accuracy: {model.score(X_train_scaled, y_train):.4f}')
print(f'Model saved to: {model_path}')

In [ ]:
# Record model artifact and complete execution
model_artifact_id = put_artifact(model_type_id, model_path,
                                  {'name': 'Wine SVM', 'version': 1, 'framework': 'scikit-learn'})

put_event(model_artifact_id, training_exec_id, 'DECLARED_OUTPUT')
update_execution(training_exec_id, 'COMPLETED')

print(f'Model artifact ID: {model_artifact_id}')
print('Model Training execution: COMPLETED')

## Stage 4: Model Evaluation

We evaluate the model on the eval dataset and record accuracy, F1, precision, and recall.

In [ ]:
# Register eval dataset artifact for evaluation
eval_data_artifact_id = put_artifact(data_type_id, './data/eval/data.csv',
                                      {'name': 'Wine dataset', 'split': 'eval', 'version': 1})

# Create evaluation execution with model + eval data as inputs
eval_exec_id = put_execution(eval_exec_type_id, 'RUNNING')
put_event(model_artifact_id, eval_exec_id, 'DECLARED_INPUT')
put_event(eval_data_artifact_id, eval_exec_id, 'DECLARED_INPUT')

print(f'Eval dataset artifact ID: {eval_data_artifact_id}')
print(f'Model Evaluation execution ID: {eval_exec_id}')

In [ ]:
# Run evaluation
X_eval = eval_df.drop('target', axis=1)
y_eval = eval_df['target']

X_eval_scaled = scaler.transform(X_eval)
y_pred = model.predict(X_eval_scaled)

metrics = {
    'accuracy': round(accuracy_score(y_eval, y_pred), 4),
    'f1_score': round(f1_score(y_eval, y_pred, average='weighted'), 4),
    'precision': round(precision_score(y_eval, y_pred, average='weighted'), 4),
    'recall': round(recall_score(y_eval, y_pred, average='weighted'), 4)
}

# Save metrics
metrics_path = './model/eval_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Evaluation on {len(X_eval)} samples:')
for name, val in metrics.items():
    print(f'  {name}: {val}')
print(f'\nMetrics saved to: {metrics_path}')

In [ ]:
# Record evaluation artifact and complete execution
eval_artifact_id = put_artifact(eval_type_id, metrics_path,
                                 {'name': 'Wine SVM Evaluation',
                                  'accuracy': metrics['accuracy'],
                                  'f1_score': metrics['f1_score'],
                                  'precision': metrics['precision'],
                                  'recall': metrics['recall']})

put_event(eval_artifact_id, eval_exec_id, 'DECLARED_OUTPUT')
update_execution(eval_exec_id, 'COMPLETED')

print(f'Evaluation artifact ID: {eval_artifact_id}')
print('Model Evaluation execution: COMPLETED')

## Setting up Context and Generating Attributions/Associations

We group all artifacts and executions under a single experiment context.

In [ ]:
# Create context type and context
expt_context_type_id = put_context_type('Experiment', {'note': 'STRING'})

expt_context_id = put_context(expt_context_type_id, 'Wine Classification Pipeline',
                               {'note': 'Data validation, anomaly detection, model training, and evaluation for Wine dataset'})

# Attributions: link artifacts to context
put_attribution(schema_artifact_id, expt_context_id)
put_attribution(anomaly_artifact_id, expt_context_id)
put_attribution(model_artifact_id, expt_context_id)
put_attribution(eval_artifact_id, expt_context_id)

# Associations: link executions to context
put_association(dv_exec_id, expt_context_id)
put_association(anomaly_exec_id, expt_context_id)
put_association(training_exec_id, expt_context_id)
put_association(eval_exec_id, expt_context_id)

print(f'Context type ID: {expt_context_type_id}')
print(f'Context ID: {expt_context_id}')
print('Attributions: Schema, Anomalies, Model, Evaluation')
print('Associations: Data Validation, Anomaly Detection, Model Training, Model Evaluation')

## Retrieving Information from the Metadata Store

Now we query the store to trace lineage — the same way you would with the real MLMD API.

In [ ]:
# Get all artifact types
print('=== Registered Artifact Types ===')
for row in cursor.execute('SELECT * FROM artifact_types').fetchall():
    print(f'  ID={row[0]}, Name={row[1]}, Properties={row[2]}')

In [ ]:
# Trace schema lineage: Schema -> Data Validation -> Dataset
print('=== Schema Lineage ===')
schema_artifacts = get_artifacts_by_type('Schema')
schema_to_inv = schema_artifacts[0]
print(f'Schema: {schema_to_inv["properties"]["name"]} (uri: {schema_to_inv["uri"]})')

# Find the execution that produced it
schema_events = get_events_by_artifact(schema_to_inv['id'])
output_event = [e for e in schema_events if e['type'] == 'DECLARED_OUTPUT'][0]

# Find inputs to that execution
exec_events = get_events_by_execution(output_event['execution_id'])
for event in exec_events:
    if event['type'] == 'DECLARED_INPUT':
        artifact = get_artifact(event['artifact_id'])
        type_name = get_artifact_type_name(artifact['type_id'])
        print(f'  Input: [{type_name}] {artifact["properties"]["name"]} (uri: {artifact["uri"]})')

In [ ]:
# Retrieve anomaly detection results
print('=== Anomaly Detection Results ===')
anomaly_artifacts = get_artifacts_by_type('Anomalies')
anomaly_to_inv = anomaly_artifacts[0]
print(f'Anomalies found: {anomaly_to_inv["properties"]["num_anomalies"]}')
print(f'Description: {anomaly_to_inv["properties"]["description"]}')

# Trace inputs
anomaly_events = get_events_by_artifact(anomaly_to_inv['id'])
anomaly_output = [e for e in anomaly_events if e['type'] == 'DECLARED_OUTPUT'][0]
anomaly_exec_events = get_events_by_execution(anomaly_output['execution_id'])

print('\nInputs to the anomaly detection execution:')
for event in anomaly_exec_events:
    if event['type'] == 'DECLARED_INPUT':
        artifact = get_artifact(event['artifact_id'])
        type_name = get_artifact_type_name(artifact['type_id'])
        print(f'  [{type_name}] {artifact["properties"]["name"]} (uri: {artifact["uri"]})')

In [ ]:
# Trace model lineage
print('=== Model Lineage ===')
model_artifacts = get_artifacts_by_type('Model')
model_to_inv = model_artifacts[0]
print(f'Model: {model_to_inv["properties"]["name"]} (framework: {model_to_inv["properties"]["framework"]})')

model_events = get_events_by_artifact(model_to_inv['id'])
model_output = [e for e in model_events if e['type'] == 'DECLARED_OUTPUT'][0]
training_events = get_events_by_execution(model_output['execution_id'])

print('\nDataset used to train the model:')
for event in training_events:
    if event['type'] == 'DECLARED_INPUT':
        artifact = get_artifact(event['artifact_id'])
        type_name = get_artifact_type_name(artifact['type_id'])
        print(f'  [{type_name}] {artifact["properties"]["name"]} split={artifact["properties"]["split"]} (uri: {artifact["uri"]})')

In [ ]:
# Trace evaluation metrics lineage
print('=== Evaluation Metrics Lineage ===')
eval_artifacts = get_artifacts_by_type('ModelEvaluation')
eval_to_inv = eval_artifacts[0]
print(f'Evaluation metrics:')
print(f'  Accuracy:  {eval_to_inv["properties"]["accuracy"]}')
print(f'  F1 Score:  {eval_to_inv["properties"]["f1_score"]}')
print(f'  Precision: {eval_to_inv["properties"]["precision"]}')
print(f'  Recall:    {eval_to_inv["properties"]["recall"]}')

eval_events = get_events_by_artifact(eval_to_inv['id'])
eval_output = [e for e in eval_events if e['type'] == 'DECLARED_OUTPUT'][0]
eval_exec_events = get_events_by_execution(eval_output['execution_id'])

print('\nInputs to the evaluation execution:')
for event in eval_exec_events:
    if event['type'] == 'DECLARED_INPUT':
        artifact = get_artifact(event['artifact_id'])
        type_name = get_artifact_type_name(artifact['type_id'])
        print(f'  [{type_name}] {artifact["properties"]["name"]} (uri: {artifact["uri"]})')

In [ ]:
# Full pipeline summary
print('=== All Artifacts in the Experiment Context ===')
for a in get_artifacts_by_context(expt_context_id):
    type_name = get_artifact_type_name(a['type_id'])
    print(f'  [{type_name}] {a["properties"]["name"]} (id: {a["id"]})')

print('\n=== All Executions in the Experiment Context ===')
for e in get_executions_by_context(expt_context_id):
    type_name = get_execution_type_name(e['type_id'])
    print(f'  [{type_name}] state: {e["state"]} (id: {e["id"]})')

In [ ]:
# Close the database connection
conn.close()
print(f'\nMetadata store saved at: {db_path}')
print('Database connection closed.')

### Wrap Up

In this notebook, you practiced using the ML Metadata concepts to track a complete ML pipeline with four stages:

1. **Data Validation** — Inferred a schema from training data
2. **Anomaly Detection** — Validated eval data against the schema to catch data drift
3. **Model Training** — Trained an SVM classifier on the Wine dataset
4. **Model Evaluation** — Computed accuracy, F1, precision, and recall metrics

All artifacts, executions, and their relationships were recorded in a **persistent SQLite-backed** metadata store, enabling full lineage tracking across sessions. The same data model (ArtifactTypes, Artifacts, ExecutionTypes, Executions, Events, Contexts, Attributions, Associations) is used by the official ML Metadata library.